# Final Policy Guardrail Grid Audit

Ce notebook reprend le script `final_policy_guardrail_grid_audit.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Balaye des politiques transparentes attention/PPE pour score live final.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Sweep transparent attention/PPE guardrail policies for the final fused risk score.
- Commande de reproduction referencee : final policy guardrail grid.
- Artefacts controles : Transparent final attention/PPE policy guardrail grid audit exists. (`runs/exp_069_final_policy_guardrail_grid/metrics/policy_guardrail_operating_summary.csv`).
- Run par defaut : `runs/exp_069_final_policy_guardrail_grid`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "final_policy_guardrail_grid_audit.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

from ml_pipeline import ROOT, safe_auc, threshold_sweep, write_json
from sequence_experiments import append_report, make_run_dir


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    return path if path.is_absolute() else ROOT / path


## Fonction `parse_grid`

Cette cellule definit `parse_grid`. Elle prepare une partie du script.

In [ ]:
def parse_grid(text):
    values = []
    for item in str(text).split(","):
        item = item.strip()
        if item:
            values.append(float(item))
    return values


## Fonction `ece_score`

Cette cellule definit `ece_score`. Elle prepare une partie du script.

In [ ]:
def ece_score(y, p, bins=10):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = max(1, len(y))
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (p >= lo) & (p <= hi) if hi == 1.0 else (p >= lo) & (p < hi)
        if mask.any():
            ece += float(mask.sum()) / total * abs(float(y[mask].mean()) - float(p[mask].mean()))
    return float(ece)


## Fonction `load_score_frames`

Cette cellule definit `load_score_frames`. Elle prepare une partie du script.

In [ ]:
def load_score_frames(score_run):
    frames = {}
    for path in sorted((score_run / "features").glob("final_scores_seed*.csv")):
        seed = int(path.stem.replace("final_scores_seed", ""))
        frames[seed] = pd.read_csv(path)
    if not frames:
        raise SystemExit(f"No final score feature files found under {score_run / 'features'}")
    return frames


## Fonction `build_policy_grid`

Cette cellule definit `build_policy_grid`. Elle prepare une partie du script.

In [ ]:
def build_policy_grid(args):
    attention_weights = parse_grid(args.attention_weights)
    ppe_weights = parse_grid(args.ppe_weights)
    interaction_weights = parse_grid(args.interaction_weights)

    variants = []
    variants.append(
        {
            "variant": "policy_sequence_only",
            "family": "sequence_only",
            "attention_weight": 0.0,
            "ppe_weight": 0.0,
            "interaction_weight": 0.0,
            "description": "d",
        }
    )

    for aw in attention_weights:
        for pw in ppe_weights:
            if aw == 0.0 and pw == 0.0:
                continue
            variants.append(
                {
                    "variant": f"mult_aw{aw:.2f}_pw{pw:.2f}".replace(".", "p"),
                    "family": "multiplicative",
                    "attention_weight": aw,
                    "ppe_weight": pw,
                    "interaction_weight": 0.0,
                    "description": "1-(1-d)*(1-aw*a)*(1-pw*p)",
                }
            )

    for aw in attention_weights:
        for pw in ppe_weights:
            for iw in interaction_weights:
                variants.append(
                    {
                        "variant": f"prior_aw{aw:.2f}_pw{pw:.2f}_iw{iw:.2f}".replace(".", "p"),
                        "family": "prior_max",
                        "attention_weight": aw,
                        "ppe_weight": pw,
                        "interaction_weight": iw,
                        "description": "max(d, clip(aw*a + pw*p + iw*a*p, 0, 1))",
                    }
                )
                variants.append(
                    {
                        "variant": f"resid_aw{aw:.2f}_pw{pw:.2f}_iw{iw:.2f}".replace(".", "p"),
                        "family": "residual_additive",
                        "attention_weight": aw,
                        "ppe_weight": pw,
                        "interaction_weight": iw,
                        "description": "clip(d + (1-d)*(aw*a + pw*p + iw*a*p), 0, 1)",
                    }
                )

    # Existing named policies are included to anchor the grid against previous reports.
    variants.extend(
        [
            {
                "variant": "existing_attention_ppe_prior",
                "family": "existing_named",
                "attention_weight": 0.25,
                "ppe_weight": 0.20,
                "interaction_weight": 0.10,
                "description": "max(d, clip(0.25*a + 0.20*p + 0.10*a*p, 0, 1))",
            },
            {
                "variant": "existing_safety_sensitive_rule",
                "family": "existing_named",
                "attention_weight": 0.45,
                "ppe_weight": 0.35,
                "interaction_weight": 0.0,
                "description": "1-(1-d)*(1-0.45*a)*(1-0.35*p)",
            },
        ]
    )
    return pd.DataFrame(variants).drop_duplicates("variant").reset_index(drop=True)


## Fonction `apply_policy`

Cette cellule definit `apply_policy`. Elle prepare une partie du script.

In [ ]:
def apply_policy(df, spec):
    d = df["final_sequence_only"].astype(float).clip(0, 1)
    a = df["attention_risk"].astype(float).clip(0, 1)
    p = df["ppe_risk"].astype(float).clip(0, 1)
    aw = float(spec["attention_weight"])
    pw = float(spec["ppe_weight"])
    iw = float(spec["interaction_weight"])
    family = spec["family"]
    if family == "sequence_only":
        return d
    if family == "multiplicative":
        return (1.0 - (1.0 - d) * (1.0 - aw * a) * (1.0 - pw * p)).clip(0, 1)
    if family == "prior_max":
        return np.maximum(d, (aw * a + pw * p + iw * a * p).clip(0, 1))
    if family == "residual_additive":
        return (d + (1.0 - d) * (aw * a + pw * p + iw * a * p)).clip(0, 1)
    if spec["variant"] == "existing_attention_ppe_prior":
        return np.maximum(d, (0.25 * a + 0.20 * p + 0.10 * a * p).clip(0, 1))
    if spec["variant"] == "existing_safety_sensitive_rule":
        return (1.0 - (1.0 - d) * (1.0 - 0.45 * a) * (1.0 - 0.35 * p)).clip(0, 1)
    raise ValueError(spec["variant"])


## Fonction `policy_floor`

Cette cellule definit `policy_floor`. Elle prepare une partie du script.

In [ ]:
def policy_floor(spec):
    aw = float(spec["attention_weight"])
    pw = float(spec["ppe_weight"])
    iw = float(spec["interaction_weight"])
    if spec["family"] in {"prior_max", "residual_additive"}:
        return {
            "score_when_only_bad_attention": min(1.0, aw),
            "score_when_only_bad_ppe": min(1.0, pw),
            "score_when_bad_attention_and_ppe": min(1.0, aw + pw + iw),
        }
    if spec["family"] == "multiplicative":
        return {
            "score_when_only_bad_attention": min(1.0, aw),
            "score_when_only_bad_ppe": min(1.0, pw),
            "score_when_bad_attention_and_ppe": min(1.0, 1.0 - (1.0 - aw) * (1.0 - pw)),
        }
    if spec["variant"] == "existing_attention_ppe_prior":
        return {
            "score_when_only_bad_attention": 0.25,
            "score_when_only_bad_ppe": 0.20,
            "score_when_bad_attention_and_ppe": 0.55,
        }
    if spec["variant"] == "existing_safety_sensitive_rule":
        return {
            "score_when_only_bad_attention": 0.45,
            "score_when_only_bad_ppe": 0.35,
            "score_when_bad_attention_and_ppe": 1.0 - (1.0 - 0.45) * (1.0 - 0.35),
        }
    return {
        "score_when_only_bad_attention": 0.0,
        "score_when_only_bad_ppe": 0.0,
        "score_when_bad_attention_and_ppe": 0.0,
    }


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(df, group_cols, metric_cols):
    rows = []
    for keys, group in df.groupby(group_cols):
        key_tuple = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_cols, key_tuple))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        for col in metric_cols:
            values = pd.to_numeric(group[col], errors="coerce")
            row[f"{col}_mean"] = float(values.mean())
            row[f"{col}_std"] = float(values.std(ddof=0))
        rows.append(row)
    return pd.DataFrame(rows)


## Fonction `select_threshold`

Cette cellule definit `select_threshold`. Elle prepare une partie du script.

In [ ]:
def select_threshold(val_sweep, policy):
    work = val_sweep.copy()
    numeric_cols = [
        "threshold",
        "window_precision",
        "window_recall",
        "window_f1",
        "danger_clip_hit_rate",
        "safe_false_alarms_per_min",
        "median_early_warning_s",
    ]
    for col in numeric_cols:
        work[col] = pd.to_numeric(work[col], errors="coerce")
    if policy == "max_hit_low_fa":
        return work.sort_values(["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"], ascending=[False, True, False]).iloc[0]
    if policy == "early_high_recall":
        eligible = work[(work["danger_clip_hit_rate"] >= 0.85) & (work["median_early_warning_s"] >= 1.0)]
        if eligible.empty:
            eligible = work[work["danger_clip_hit_rate"] >= 0.85]
        if eligible.empty:
            eligible = work
        return eligible.sort_values(["danger_clip_hit_rate", "median_early_warning_s", "safe_false_alarms_per_min", "window_precision"], ascending=[False, False, True, False]).iloc[0]
    if policy == "balanced_f1":
        return work.sort_values(["window_f1", "danger_clip_hit_rate", "safe_false_alarms_per_min"], ascending=[False, False, True]).iloc[0]
    if policy == "precision_guard":
        eligible = work[work["window_precision"] >= 0.50]
        if eligible.empty:
            eligible = work
        return eligible.sort_values(["danger_clip_hit_rate", "window_f1", "safe_false_alarms_per_min"], ascending=[False, False, True]).iloc[0]
    if policy == "low_false_alarm":
        eligible = work[work["safe_false_alarms_per_min"] <= 3.0]
        if eligible.empty:
            eligible = work
        return eligible.sort_values(["danger_clip_hit_rate", "window_f1", "window_precision", "safe_false_alarms_per_min"], ascending=[False, False, False, True]).iloc[0]
    raise ValueError(policy)


## Fonction `apply_selected_threshold`

Cette cellule definit `apply_selected_threshold`. Elle prepare une partie du script.

In [ ]:
def apply_selected_threshold(test_sweep, threshold):
    work = test_sweep.copy()
    work["threshold_distance"] = (pd.to_numeric(work["threshold"], errors="coerce") - float(threshold)).abs()
    return work.sort_values("threshold_distance").iloc[0]


## Fonction `evaluate_variant`

Cette cellule definit `evaluate_variant`. Elle prepare une partie du script.

In [ ]:
def evaluate_variant(df, spec, seed, policies, persistence_windows, splits):
    work = df.copy()
    score_col = "policy_score"
    work[score_col] = apply_policy(work, spec)
    metric_rows = []
    sweep_frames = []
    for split in splits:
        split_df = work[work["split"].eq(split)].copy()
        y = split_df["danger_within_1.0s"].astype(int).to_numpy()
        p = split_df[score_col].astype(float).to_numpy()
        sweep = threshold_sweep(split_df.rename(columns={score_col: "risk"}), "risk", 1.0, split, persistence_windows=persistence_windows)
        sweep["repeat_seed"] = seed
        sweep["variant"] = spec["variant"]
        sweep["family"] = spec["family"]
        sweep_frames.append(sweep)
        metric_rows.append(
            {
                "repeat_seed": seed,
                "variant": spec["variant"],
                "family": spec["family"],
                "split": split,
                "n": int(len(split_df)),
                "positives": int(y.sum()),
                "average_precision": safe_auc(average_precision_score, y, p),
                "roc_auc": safe_auc(roc_auc_score, y, p),
                "brier": float(brier_score_loss(y, p)) if len(np.unique(y)) > 1 else np.nan,
                "ece_10bin": ece_score(y, p),
            }
        )

    sweep_df = pd.concat(sweep_frames, ignore_index=True)
    selected_rows = []
    val_sweep = sweep_df[sweep_df["split"].eq("val")]
    test_sweep = sweep_df[sweep_df["split"].eq("test")]
    for policy in policies:
        selected = select_threshold(val_sweep, policy)
        exact = apply_selected_threshold(test_sweep, float(selected["threshold"]))
        selected_rows.append(
            {
                "repeat_seed": seed,
                "variant": spec["variant"],
                "family": spec["family"],
                "policy": policy,
                "selected_threshold": float(selected["threshold"]),
                "test_hit_rate": float(exact["danger_clip_hit_rate"]),
                "test_false_alarms_per_min": float(exact["safe_false_alarms_per_min"]),
                "test_window_precision": float(exact["window_precision"]),
                "test_window_recall": float(exact["window_recall"]),
                "test_window_f1": float(exact["window_f1"]),
                "test_median_early_warning_s": exact["median_early_warning_s"],
            }
        )
    return metric_rows, selected_rows, sweep_df


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, grid, metric_summary, operating_summary):
    test_metrics = metric_summary[metric_summary["split"].eq("test")].copy()
    test_metrics = test_metrics.merge(grid, on=["variant", "family"], how="left")
    ops = operating_summary.merge(grid, on=["variant", "family"], how="left")
    for col in ["test_median_early_warning_s_mean", "test_false_alarms_per_min_mean", "test_window_precision_mean", "test_hit_rate_mean"]:
        ops[col] = pd.to_numeric(ops[col], errors="coerce")
    ops["gate_count"] = (
        (ops["test_hit_rate_mean"] >= 0.85).astype(int)
        + (ops["test_false_alarms_per_min_mean"] <= 5.0).astype(int)
        + (ops["test_window_precision_mean"] >= 0.40).astype(int)
        + (ops["test_median_early_warning_s_mean"] >= 1.0).astype(int)
    )
    ops["policy_rank"] = (
        ops["gate_count"].astype(float)
        + 0.75 * ops["test_hit_rate_mean"].fillna(0)
        + 0.25 * ops["test_window_precision_mean"].fillna(0)
        - 0.035 * ops["test_false_alarms_per_min_mean"].fillna(20).clip(upper=20)
        + 0.10 * ops["test_median_early_warning_s_mean"].fillna(0).clip(upper=2.0)
    )
    top_ap = test_metrics.sort_values("average_precision_mean", ascending=False).head(12)
    top_policy = ops.sort_values("policy_rank", ascending=False).head(16)
    strong_ppe = ops[ops["score_when_only_bad_ppe"] >= 0.50].sort_values("policy_rank", ascending=False).head(10)

    lines = ["# Final Policy Guardrail Grid Audit", ""]
    lines.append("This audit sweeps transparent attention/PPE guardrail weights over the existing final trajectory, attention, and PPE score streams. No final fusion weights are learned here; the grid measures human-specified safety policies.")
    lines.append("")
    lines.append("## Top Test AP Policies")
    lines.append("")
    lines.append("| rank | variant | family | aw | pw | iw | AP | ROC AUC | Brier | ECE |")
    lines.append("|---:|---|---|---:|---:|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(top_ap.iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['variant']} | {row['family']} | {row['attention_weight']:.2f} | {row['ppe_weight']:.2f} | {row['interaction_weight']:.2f} | "
            f"{row['average_precision_mean']:.3f} | {row['roc_auc_mean']:.3f} | {row['brier_mean']:.3f} | {row['ece_10bin_mean']:.3f} |"
        )
    lines.append("")
    lines.append("## Best Validation-Selected Operating Policies")
    lines.append("")
    lines.append("| rank | variant | family | policy | aw | pw | iw | threshold | AP | hit | FA/min | precision | F1 | median early s | gates |")
    lines.append("|---:|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(top_policy.iterrows(), start=1):
        ap_row = test_metrics[test_metrics["variant"].eq(row["variant"])].iloc[0]
        lines.append(
            f"| {rank} | {row['variant']} | {row['family']} | {row['policy']} | {row['attention_weight']:.2f} | {row['ppe_weight']:.2f} | {row['interaction_weight']:.2f} | "
            f"{row['selected_threshold_mean']:.2f} | {ap_row['average_precision_mean']:.3f} | {row['test_hit_rate_mean']:.3f} | {row['test_false_alarms_per_min_mean']:.3f} | "
            f"{row['test_window_precision_mean']:.3f} | {row['test_window_f1_mean']:.3f} | {row['test_median_early_warning_s_mean']:.3f} | {int(row['gate_count'])}/4 |"
        )
    lines.append("")
    lines.append("## Strong PPE Guardrail Candidates")
    lines.append("")
    lines.append("These rows require a policy floor of at least `0.50` when PPE/blouse is badly worn even if trajectory danger is low.")
    lines.append("")
    lines.append("| rank | variant | family | policy | PPE floor | attention floor | threshold | AP | hit | FA/min | precision | median early s | gates |")
    lines.append("|---:|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(strong_ppe.iterrows(), start=1):
        ap_row = test_metrics[test_metrics["variant"].eq(row["variant"])].iloc[0]
        lines.append(
            f"| {rank} | {row['variant']} | {row['family']} | {row['policy']} | {row['score_when_only_bad_ppe']:.2f} | {row['score_when_only_bad_attention']:.2f} | "
            f"{row['selected_threshold_mean']:.2f} | {ap_row['average_precision_mean']:.3f} | {row['test_hit_rate_mean']:.3f} | {row['test_false_alarms_per_min_mean']:.3f} | "
            f"{row['test_window_precision_mean']:.3f} | {row['test_median_early_warning_s_mean']:.3f} | {int(row['gate_count'])}/4 |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- This is a policy-grid audit, not a learned fusion model.")
    lines.append("- The relevant choice is not only AP; it is the named operating mode plus threshold, hit rate, false alarms, precision, and early-warning time.")
    lines.append("- Stronger PPE/attention priors deliberately make the score fire earlier and can be selected if the demonstration goal is conservative safety behavior before danger-zone entry.")
    lines.append("")
    lines.append("## Artifacts")
    lines.append("")
    lines.append(f"- Policy grid: `{run_dir / 'metrics' / 'policy_guardrail_grid.csv'}`")
    lines.append(f"- Metric summary: `{run_dir / 'metrics' / 'policy_guardrail_metric_summary.csv'}`")
    lines.append(f"- Operating summary: `{run_dir / 'metrics' / 'policy_guardrail_operating_summary.csv'}`")
    summary_path = run_dir / "final_policy_guardrail_grid_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Final Policy Guardrail Grid Audit", f"- Summary: `{summary_path}`")
    return summary_path


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    score_run = resolve(args.score_run)
    run_dir = make_run_dir(args.run_name)
    frames = load_score_frames(score_run)
    grid = build_policy_grid(args)
    floor_rows = [policy_floor(row) for _, row in grid.iterrows()]
    grid = pd.concat([grid, pd.DataFrame(floor_rows)], axis=1)
    policies = ["max_hit_low_fa", "early_high_recall", "balanced_f1", "precision_guard", "low_false_alarm"]
    write_json(
        run_dir / "config.json",
        {
            "score_run": str(score_run),
            "seeds": sorted(frames),
            "n_policy_variants": int(len(grid)),
            "persistence_windows": args.persistence_windows,
            "attention_weights": parse_grid(args.attention_weights),
            "ppe_weights": parse_grid(args.ppe_weights),
            "interaction_weights": parse_grid(args.interaction_weights),
            "policies": policies,
            "splits": args.splits,
        },
    )
    grid.to_csv(run_dir / "metrics" / "policy_guardrail_grid.csv", index=False)

    metric_rows = []
    selected_rows = []
    sweep_rows = []
    for seed, df in frames.items():
        for _, spec in grid.iterrows():
            metrics, selected, sweeps = evaluate_variant(df, spec, seed, policies, args.persistence_windows, args.splits)
            metric_rows.extend(metrics)
            selected_rows.extend(selected)
            if args.keep_sweeps:
                sweep_rows.append(sweeps)

    metrics = pd.DataFrame(metric_rows)
    selected = pd.DataFrame(selected_rows)
    metrics.to_csv(run_dir / "metrics" / "policy_guardrail_metrics.csv", index=False)
    selected.to_csv(run_dir / "metrics" / "policy_guardrail_validation_selected.csv", index=False)
    if args.keep_sweeps and sweep_rows:
        pd.concat(sweep_rows, ignore_index=True).to_csv(run_dir / "metrics" / "policy_guardrail_threshold_sweeps.csv", index=False)

    metric_summary = summarize(metrics, ["variant", "family", "split"], ["average_precision", "roc_auc", "brier", "ece_10bin"])
    operating_summary = summarize(
        selected,
        ["variant", "family", "policy"],
        [
            "selected_threshold",
            "test_hit_rate",
            "test_false_alarms_per_min",
            "test_window_precision",
            "test_window_recall",
            "test_window_f1",
            "test_median_early_warning_s",
        ],
    )
    metric_summary.to_csv(run_dir / "metrics" / "policy_guardrail_metric_summary.csv", index=False)
    operating_summary.to_csv(run_dir / "metrics" / "policy_guardrail_operating_summary.csv", index=False)
    write_summary(run_dir, grid, metric_summary, operating_summary)
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Sweep transparent attention/PPE guardrail policies for the final fused risk score.")
    parser.add_argument("--score-run", default="runs/exp_039_final_aggregated_score")
    parser.add_argument("--run-name", default="exp_069_final_policy_guardrail_grid")
    parser.add_argument("--attention-weights", default="0.00,0.15,0.30,0.45,0.60,0.75")
    parser.add_argument("--ppe-weights", default="0.00,0.20,0.35,0.50,0.65,0.80")
    parser.add_argument("--interaction-weights", default="0.00,0.10,0.20,0.30")
    parser.add_argument("--persistence-windows", type=int, default=2)
    parser.add_argument("--splits", nargs="+", default=["val", "test"], choices=["train", "val", "test"])
    parser.add_argument("--keep-sweeps", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_069_final_policy_guardrail_grid_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["final_policy_guardrail_grid_audit.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
